# Блок 4 • Занятие 1 — выборка, статистики, устойчивость к выбросам (TODO)
**Дата:** 2026-03-02  
**Формат:** мини-теория → практика в коде → автопроверка (`assert`) → Run all  
**Цель занятия:** научиться считать базовые статистики по выборке и понимать разницу “параметр vs статистика”.

---

## Мини-теория (коротко, но по делу)

### 1) Что такое выборка
- **Генеральная совокупность** — “все возможные данные” (обычно недоступно целиком).
- **Выборка** — часть данных, которую мы реально собрали (из файла/БД/API).  
В дипломе вы почти всегда работаете именно с **выборкой**.

### 2) Параметр vs статистика
- **Параметр** — истинная характеристика совокупности (например, “реальное среднее по всем пользователям”).
- **Статистика** — оценка по выборке (например, среднее по вашим 500 записям).

### 3) Главные статистики
- `mean` — среднее (чувствительно к выбросам)
- `median` — медиана (устойчивее к выбросам)
- `variance/std` — разброс (вариация, стандартное отклонение)

### 4) Почему это нужно для диплома
Это ваш “модуль качества данных”:
- увидеть выбросы и шум
- выбрать правильные метрики
- объяснить пользователю/заказчику, почему вы доверяете данным

---

## Как работать
- Запускайте ячейки сверху вниз, или **Runtime → Run all**
- В конце должно быть: **✅ BLOCK04 LESSON01: all tests passed**

---

## Задача урока (перенос в VS Code)
В конце занятия перенесите функции в файл проекта:
- `src/math_stats.py`: `mean`, `median`, `variance_sample`, `std_sample`, `trimmed_mean`  


---
## Ячейка 1/10 — данные как список чисел, `len`, `print`

**Что изучаем здесь**
- `list[float]` — список чисел (выборка)
- `len(x)` — размер выборки `n`
- `print(...)` — вывести результат


In [114]:
sample = [5, 4, 12, 1, 3, 10, 5, 7, 3, 4, 9, 14, 16, 20, 2, 3, 8]

print("sample:", sample)
print("n =", len(sample))


sample: [5, 4, 12, 1, 3, 10, 5, 7, 3, 4, 9, 14, 16, 20, 2, 3, 8]
n = 17


---
## Ячейка 2/10 — минимум/максимум: `min`, `max`, `sorted`

**Что изучаем здесь**
- `min(values)` и `max(values)` — крайние значения
- `sorted(values)` — сортировка (нужна для медианы)


In [115]:
print("min =", min(sample))
print("max =", max(sample))
print("sorted =", sorted(sample))


min = 1
max = 20
sorted = [1, 2, 3, 3, 3, 4, 4, 5, 5, 7, 8, 9, 10, 12, 14, 16, 20]


---
## Ячейка 3/10 — среднее: `sum`, `len`, функция `mean`

**Термин**
- Среднее (mean) = сумма / количество


In [116]:
def mean(values: list[float]) -> float:
    """Среднее арифметическое. Требует непустой список."""
    if len(values) == 0: 
        raise ValueError("Нельзя вычислить среднее для пустого списка")
    return sum(values) / len(values)


print("mean =", mean(sample))


mean = 7.411764705882353


---
## Ячейка 4/10 — медиана: `sorted`, индексы, чёт/нечёт

**Термин**
- Медиана — “середина” отсортированных данных.


In [117]:
def median(values: list[float]) -> float:
    """Медиана. Требует непустой список."""
    if not values:
        raise ValueError("median: empty list")
    sorted_values = sorted(values)
    n = len(sorted_values)
    middle = n // 2
    if n % 2 == 1:
        # Для нечетного n: элемент посередине
        return float(sorted_values[middle])
    else:
       # Для четного n: среднее двух центральных
        return (sorted_values[middle - 1] + sorted_values[middle]) / 2
    

print("median =", median(sample))


median = 5.0


---
## Ячейка 5/10 — выборочная дисперсия: сумма квадратов отклонений

Формула: `sum((x-mean)^2) / (n-1)`


In [118]:
def variance_sample(values: list[float]) -> float:
    """Выборочная дисперсия (деление на n-1)."""
    n = len(values)
    if n < 2:
        raise ValueError("Нужно минимум 2 элемента")
    m = mean(values)
    return sum((x - m) ** 2 for x in values) / (n - 1)

print("variance_sample =", variance_sample(sample))


variance_sample = 29.38235294117647


---
## Ячейка 6/10 — стандартное отклонение: корень из дисперсии

`std = variance ** 0.5`


In [119]:
def std_sample(values: list[float]) -> float:
    """Выборочное стандартное отклонение."""     
    var = variance_sample(values)
    return var ** 0.5

print("std_sample =", std_sample(sample))


std_sample = 5.420549136496825


---
## Ячейка 7/10 — выбросы: сравним mean и median “до/после”

Выброс добавляем в новый список, исходный не меняем.


In [120]:
def with_outlier(values: list[float], outlier: float) -> list[float]:
    """Вернуть новую выборку, добавив выброс (не меняем исходный список)."""
    return values + [outlier]

sample_out = with_outlier(sample, 100)

print("mean(before) =", round(mean(sample), 3), "median(before) =", median(sample))
print("mean(after)  =", round(mean(sample_out), 3), "median(after)  =", median(sample_out))


mean(before) = 7.412 median(before) = 5.0
mean(after)  = 12.556 median(after)  = 6.0


---
## Ячейка 8/10 — trimmed mean: устойчивое среднее

Убираем `k` самых маленьких и `k` самых больших значений.


In [121]:
def trimmed_mean(values: list[float], k: int = 1) -> float:
    """Усечённое среднее: убрать k минимальных и k максимальных."""
    n = len(values)
    if not values:
        raise ValueError("Список не может быть пустым")
    if 2 * k >= n:
        raise ValueError("k слишком большое")
    sorted_values = sorted(values)
    core = sorted_values[k:n-k] 
    return mean(core)

print("trimmed_mean(before) =", round(trimmed_mean(sample, k=1), 3))
print("trimmed_mean(after)  =", round(trimmed_mean(sample_out, k=1), 3))


trimmed_mean(before) = 7.0
trimmed_mean(after)  = 7.812


---
## Ячейка 9/10 — функция `describe`: мини-отчёт в словаре (dict)

Это удобно для логирования, БД и отчётов в дипломе.


In [122]:
def describe(values: list[float]) -> dict:
    """Короткое описание выборки (как мини-отчёт)."""
    return {
        "n": len(values),
        "min": min(values) if values else None,
        "max": max(values) if values else None,
        "mean": mean(values) if values else None,
        "median": median(values) if values else None,
        "std": std_sample(values) if len(values) >= 2 else None,
    }

print("describe(sample) =", describe(sample))
print("describe(sample_out) =", describe(sample_out))


describe(sample) = {'n': 17, 'min': 1, 'max': 20, 'mean': 7.411764705882353, 'median': 5.0, 'std': 5.420549136496825}
describe(sample_out) = {'n': 18, 'min': 1, 'max': 100, 'mean': 12.555555555555555, 'median': 6.0, 'std': 22.447906288383024}


---
## Ячейка 10/10 — автопроверка (`assert`)

В solved-версии все тесты должны пройти.  
В TODO-версии тесты пройдут после того, как вы замените `# TODO` на рабочий код.


In [123]:
# =========================
# BLOCK 04 — LESSON 01 TESTS (НЕ МЕНЯТЬ)
# =========================

def approx(a: float, b: float, eps: float = 1e-9) -> bool:
    return abs(a - b) <= eps

def run_all_tests():
    s = [5, 4, 12, 1, 3, 10, 5, 7, 3, 4, 9, 14, 16, 20, 2, 3, 8]

    assert len(s) == 17
    assert min(s) == 1
    assert max(s) == 20

    assert approx(mean(s), 7.411764705882353)
    assert median(s) == 5.0

    expected_var = 29.38235294117647  
    assert approx(variance_sample(s), expected_var)
    
    expected_std = expected_var ** 0.5  # 5.420110552835053
    assert approx(std_sample(s), expected_std)

    s2 = with_outlier(s, 100)
    assert len(s2) == 18
    assert approx(mean(s2), (sum(s) + 100) / 18)
    assert median(s2) == 6.0

    assert approx(trimmed_mean([1, 2, 3, 100], k=1), 2.5)
    assert trimmed_mean([1, 2, 3, 4, 100, 101], k=1) == mean([2, 3, 4, 100])

    d = describe(s)
    assert d["n"] == 17
    assert d["min"] == 1 and d["max"] == 20
    assert approx(d["mean"], 7.411764705882353)
    assert d["median"] == 5.0
    assert d["std"] is not None

    print("✅ BLOCK04 LESSON01: all tests passed")

run_all_tests()

✅ BLOCK04 LESSON01: all tests passed
